# Download and import needed packages

In [28]:
# !pip install metapub

In [29]:
# !pip install lxml[html_clean]

In [30]:
# !pip install ratelimit

In [31]:
# !pip install lxml

In [32]:
# !pip install html5lib

In [33]:
# !pip install selenium

In [34]:
# !apt-get update 

In [35]:
# !pip install requests_html

In [36]:
# !pip install pytest-playwright

# Specify global variables
Define any regular expressions and dictionaries that will be needed later on

In [37]:
# Process text files with Regular Expressions
import re

# Retreive files from online
import urllib.request

# Access files inside the folder directory
from pathlib import Path

# Create dictionaries with lists as values
from collections import defaultdict

In [38]:
# Specify NCBI API KEY
%env NCBI_API_KEY="6667a919224612da1287d74ff0d3f7b5e208"

env: NCBI_API_KEY="6667a919224612da1287d74ff0d3f7b5e208"


In [39]:
# Regex format of DOI links, mutations, blocks, and literature type
doi_pattern = r'https:\/\/doi\.org\/[\w/.-]+'
mutation_pattern = r'(.*)?\n\n'
block_pattern = r'(?:(?<=\n\n)|^)(.+?)(?=\n\n|\Z)'
literature_pattern = r'(?<=\[)(.*?)(?=\])'
url_pattern = r'https:\/\/[^\s]+'

In [40]:
# Dictionary of doi to BioC JSON files 
publication_bioc = {}
grey_bioc = {}
rxiv_bioc = {}

# Dictionary of doi to pokay mutation summaries from that article
publication_key = defaultdict(list)
grey_key = defaultdict(list)
rxiv_key = defaultdict(list)

# Read through Pokay directory and categorize by article type
Iterate through the Pokay data directory and sort the entries into their respective dictionaries depending on if they are
- Journal article,
- Preprint, or
- Grey literature. 

In [41]:
# Retreive all files from Pokay directory
directory = '/home/david.yang1/autolit/litcovid/data/pokay'
files = Path(directory).glob('*/*')

In [42]:
# Iterate through all files in the pokay directory
for file in files:
    with open(file, 'r') as f:
    
        # Read file
        file_contents = f.read()

        # Find all mutations
        # mutations = re.findall(mutation_pattern, file_contents)

        # Find all text blocks
        text_blocks = re.findall(block_pattern, file_contents, re.DOTALL)

        # Iterate through all text blocks
        for text in text_blocks:
            
            # Define the article type
            article_type = re.findall(literature_pattern, text)
            
            # If no article type provided, check format of the link
            if len(article_type)==0:
                url = re.search(url_pattern, text)
                if url:
                    doi = re.search(doi_pattern, url.group())
                    if doi:
                        rxiv_key[doi.group()].append(text)
                        rxiv_bioc[doi.group()] = None
                    else:
                        grey_key[url.group()].append(text)
                        grey_bioc[url.group()] = None
                continue

            if "Journal publication" in article_type:
                # Search for the DOI of the publication
                doi = re.search(doi_pattern, text).group()
                publication_key[doi].append(text)
                publication_bioc[doi] = None
                
            elif "Preprint" in article_type[-1]:
                # Check if new DOI is provided
                doi = re.search(doi_pattern, article_type[-1])

                # Check if Rxiv is now published
                if doi is not None:
                    publication_key[doi.group()].append(text)
                    publication_bioc[doi.group()] = None

                # Store as Rxiv
                else:
                    doi = re.search(doi_pattern, text)
                    # DOI link provided
                    if doi is not None:
                        rxiv_key[doi.group()].append(text)
                        rxiv_bioc[doi.group()] = None
                    # DOI link not provided
                    else: 
                        rxiv_key[re.search(url_pattern, text).group()].append(text)
                        rxiv_bioc[re.search(url_pattern, text).group()] = None

            # Check if the article is grey literature
            elif article_type[-1] == "Grey literature":
                # Search for url link
                url = re.search(url_pattern, text)
                if url is not None:
                    grey_key[url.group()].append(text)
                    grey_bioc[url.group()] = None
            
            # All other groups categorize as grey literature
            else:
                url = re.search(url_pattern, text)
                if url is not None:
                    grey_key[url.group()].append(text)
                    grey_bioc[url.group()] = None

# Obtain BioC JSON files for the links

**get_pubtator_bioc_json(id)**: retrieves the BioC JSON files using the Pubtator API with PMID. Returns None if the API call fails.

**get_pmid(doi)**: converts the papers doi to pmid. Returns None if it cannot be converted. 


In [16]:
# Convert DOI to PMID
from metapub.convert import doi2pmid

# BioC JSON API calls
import requests

# Limit the number of API calls to 10 per second
from ratelimit import limits, sleep_and_retry

In [17]:
# Obtain BioC JSON file from PMID or PMC with a maximum of 3 API calls per second

@sleep_and_retry
@limits(calls=3, period=1)
def get_pubtator_bioc_json(id):
    # API link for BioC
    url = "https://www-ncbi-nlm-nih-gov.ezproxy.lib.ucalgary.ca/research/bionlp/RESTful/pmcoa.cgi/BioC_json/" + str(id) + "/unicode"
    bioc = requests.get(url, allow_redirects=True)

    # Failed to grab BioC
    if bioc.status_code != 200:
        raise ConnectionError('could not download {}\nerror code: {}'.format(url, bioc.status_code))
        return None
    
    return bioc.content

In [52]:
# Obtain PMID ID from DOI link with a maximum of 3 API calls per second

@sleep_and_retry
@limits(calls=3, period=1)
def get_pmid(doi):
    # pmid = doi2pmid(doi)
    doi_part = doi.split('doi.org/')[-1]

    # Api link for paper details
    api_link = 'https://www-ncbi-nlm-nih-gov.ezproxy.lib.ucalgary.ca/pmc/utils/idconv/v1.0/?tool=doi2pmid&email=david.yang1@ucalgary.ca&ids=' + doi_part
    paper = requests.get(api_link)
    soup = BeautifulSoup(paper.content, "xml")
        
    pmid = soup.find('record')['pmid']
    return pmid
    

# Web scraper to obtain the new DOI for the published article

In [19]:
# Scrape PMID from online
from bs4 import BeautifulSoup
import html5lib
from requests_html import HTMLSession
from selenium import webdriver

In [20]:
# Convert missing links into PMID
def scrape_PMID(url):   
    html = get_html(url, '#content')
    print(html)
    soup = BeautifulSoup(html, "lxml")
    # paper_title = soup.find(attrs={'id': 'page-title'})
    print(soup.prettify())
    # publication_link = soup.find("div", {"id": "pub_link"})
    # print(publication_link)
    # print(paper_title)
    return None

In [21]:
# publication_key

In [22]:
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time

In [23]:
# Function to grab html information from the url
async def get_html(url, selector, sleep=5, retries=5):
    html = None
    for i in range(1, retries + 1):
        time.sleep(sleep * i)

        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch()
                page = await browser.new_page()
                await page.goto(url)
                print(await page.title())
                html = await page.inner_html(selector)

        except PlaywrightTimeout:
            print(f"Timeout error on {url}")
            continue
        else:
            break

    return html

In [24]:
# scrape_PMID('https://doi.org/10.1101/2021.02.05.430003')

In [25]:
# scrape_PMID('https://doi.org/10.1101/2021.05.15.444222')

In [26]:
from requests_html import HTMLSession
from requests_html import AsyncHTMLSession
import nest_asyncio

In [27]:
asession = AsyncHTMLSession()
r = await asession.get('https://doi.org/10.1101/2021.05.15.444222')
await r.html.arender()
# resp=r.html.raw_html

BrowserError: Browser closed unexpectedly:


In [ ]:
nest_asyncio.apply()

session = HTMLSession()

page = session.get('https://doi.org/10.1101/2021.05.15.444222')

page.html.render()

In [ ]:
# r.content

In [ ]:
soup = BeautifulSoup(html.content, "lxml")
print(soup.prettify())

In [ ]:
with open("output1.html", "w") as file:
    file.write(str(soup))

# Download ArXiv XML (JATS) files of the articles for conversion
https://github.com/PeerJ/jats-conversion/blob/master/src/data/xsl/jats-to-pubmed.xsl

**get_arxiv_details(doi)**: grab details of papers from Rxiv using Rxiv API with DOI url. Return None if API call fails. 

**get_arxiv_pmid(doi)**: grab PMID for papers from Rxiv using DOI url.

In [58]:
import json

In [59]:
@sleep_and_retry
@limits(calls=1, period=1)
def get_arxiv_details(doi):
    doi_part = doi.split('doi.org/')[-1]
    api_link = 'https://api.biorxiv.org/details/biorxiv/' + doi_part
    preprint_details = requests.get(api_link)
    
    if preprint_details.status_code != 200:
        raise ConnectionError('could not download {}\nerror code: {}'.format(api_link, preprint_details.status_code))
        return None
    
    return preprint_details.content

In [60]:
@sleep_and_retry
@limits(calls=3, period=1)
def get_arxiv_pmid(doi):
    details = get_arxiv_details(doi).decode('utf-8')
    pmid = None
    
    # Load the JSON data
    data = json.loads(details)

    title = data['collection'][0]['title']
    modified_title = title.replace(" ", "%20")
    pubmed_link = "http://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&retmode=json&retmax=1000&term=" + modified_title + "&field=title"
    
    data_json = requests.get(pubmed_link).content.decode('utf-8')
    data = json.loads(data_json)
    pmid = data['esearchresult']['idlist'][0]

    return pmid

In [ ]:
pmid = get_arxiv_pmid('https://doi.org/10.1101/2021.05.15.444222')
print(pmid)

In [ ]:
details = get_arxiv_details('https://doi.org/10.1101/2021.05.15.444222').decode('utf-8')
print(details)

In [ ]:
data = json.loads(details)
print(data)

In [ ]:
print(data['collection'][0]['published'])

"published" in data['collection'][0]

In [ ]:
def get_arxiv_published_doi(details):
    data = details.decode('utf-8')
    data = json.loads(details)

    # Check if it is published
    if "published" in data["collection"][0]:
        doi = "https://doi.org/" + data['collection'][0]['published']
        return doi
    else:
        return None 

In [73]:
def process_arxiv_papers(doi):
    details = get_arxiv_details(doi).decode('utf-8')

    jats_xml = None
    # Load the JSON data
    data = json.loads(details)

    # Check if it is published
    if "published" in data["collection"][0]:
        doi = "https://doi.org/" + data['collection'][0]['published']
        pmid = get_pmid(doi)
        bioc = get_pubtator_bioc_json(pmid)
    # For unpublished preprints, extract the URL from the "jatsxml" field
    else:
        jatsxml_url = data['collection'][0]['jatsxml']
        jats_xml = requests.get(jatsxml_url).content.decode('utf-8')
        
    return jats_xml
        

In [ ]:
pmid = get_pmid(doi)
bioc = get_pubtator_bioc_json(pmid)

In [ ]:
bioc = process_arxiv_papers('https://doi.org/10.1101/2021.05.15.444222')

In [ ]:
print(publication_bioc)

In [ ]:
api_response = demo.content
api_response_str = api_response.decode('utf-8')

# Load the JSON data
data = json.loads(api_response_str)

# Extract the URL from the "jatsxml" field
jatsxml_url = data['collection'][0]['jatsxml']
doi = data['collection'][0]['doi']
print(jatsxml_url)

In [ ]:
print(demo.content)

In [ ]:
doi = "https://doi.org/" + data['collection'][0]['published']
print(doi)

In [ ]:
soup = BeautifulSoup(demo.content, "lxml")
print(soup.prettify())

In [ ]:
jats_xml = requests.get(jatsxml_url)

In [ ]:
jats_xml = jats_xml.content

In [ ]:
jats_xml = jats_xml.decode("utf-8")

In [ ]:
jats_xml

In [ ]:
with open("output.xml", "w") as file:
    file.write(jats_xml)

In [ ]:
# print(jats_xml.content)

# Convert JATS XML file to HTML

Using XSLT conversions

https://pypi.org/project/bioconverters/

https://github.com/ncbi/PubReader
https://github.com/lfurrer/bconv?tab=readme-ov-file


In [100]:
import lxml.etree as ET

In [ ]:
def convert_jatsxml_to_html(input_file, output_file):
    dom = ET.parse(input_file)
    xslt = ET.parse('data/jats-conversion/src/data/xsl/jats-to-html.xsl')
    transform = ET.XSLT(xslt)
    newdom = transform(dom)
    newdom.write_output(output_file)

In [ ]:
convert_jatsxml_to_html("output.xml", "results.html")

In [ ]:
def convert_jatsxml_to_pubmed(input_file, output_file):
    dom = ET.parse(input_file)
    xslt = ET.parse('data/jats-conversion/src/data/xsl/jats-to-pubmed.xsl')
    transform = ET.XSLT(xslt)
    newdom = transform(dom)
    newdom.write_output(output_file)

In [ ]:
convert_jatsxml_to_pubmed("output.xml", "pubmed.xml")

In [99]:
def convert_jatsxml_to_biocxml(input_file, output_file):
    dom = ET.parse(input_file)
    xslt = ET.parse('jats_to_bioc.xsl')
    transform = ET.XSLT(xslt)
    newdom = transform(dom)
    newdom.write_output(output_file)

In [101]:
convert_jatsxml_to_biocxml("input_jats.xml", "test_bioc.xml")

XMLSyntaxError: Namespace prefix bioc on collection is not defined, line 10, column 25 (jats_to_bioc.xsl, line 10)

# Convert HTML to BioC JSON
https://github.com/omicsNLP/Auto-CORPus

In [ ]:
import subprocess

In [ ]:
result = subprocess.run(["python", "data/Auto-CORPus/setup.py"], shell=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
result = subprocess.run(["python", "data/Auto-CORPus/run_app.py", "-c", "configs/config_pmc.json", "-t", "testing_output",
                         "-f", "home/david.yang1/autolit/litcovid/results.html", "-o", "JSON"], 
                        shell=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Convert HTML file to BioC JSON file
def convert_html_to_bioc_json(input_file, output_file):
    

In [ ]:
grey_bioc

# Convert JATS XML to BioC XML
Use the tool XSLTGen

# Convert links in the dictionaries to BioC JSON format

Conversion possibilities include: 
- Journal DOI to BioC JSON
- Preprint DOI to Jats XML to HTML to BioC JSON
- Grey literature HTML to BioC JSON

In [53]:
cannot_convert = 0
total = 0

# Convert the DOI links to PMID and then grab them in BioC JSON format
for key in publication_bioc:
    total += 1
    pmid = None

    if total == 20:
        break
    
    try:
        # Convert to PMID
        pmid = get_pmid(key)
    except:
        print(key)
        cannot_convert += 1
        

    # Retreive BioC JSON if conversion successful
    # if pmid:
    #     grey_bioc[key] = get_pubtator_bioc_json(pmid)

    # Convert manually if unsucessful
    # else:
        # Convert into PMID
    #     pmid = scrape_PMID(key)

        # Could not scrape PMID from PubMed, therefore generate BioC JSON
    #     if pmid is None:
            # Make your own BioC JSON file instead
    #         json = make_bioc_json(key)
            
        # Retreive BioC JSON
    #     grey_bioc[key]=get_bioc_json(pmid)

In [67]:
cannot_convert = 0
total = 0

# Convert the DOI links to PMID and then grab them in BioC JSON format
for key in rxiv_bioc:
    total += 1
    pmid = None

    if total == 20:
        break
    
    try:
        # Convert to PMID
        pmid = get_arxiv_pmid(key)
        print(key)
        print(pmid)
        print(" ")
    except:
        cannot_convert += 1

https://doi.org/10.1101/2021.02.05.430003
33564768
 
https://doi.org/10.1101/2021.03.24.436850
33791700
 
https://doi.org/10.1101/2021.05.13.444010
34031659
 
https://doi.org/10.1101/2021.05.03.442455
33972942
 
https://doi.org/10.1101/2021.03.31.437925
34210893
 


In [68]:
print(cannot_convert)

14


In [81]:
doi = 'https://doi.org/10.1101/2021.02.05.430003'
details = get_arxiv_details(doi).decode('utf-8')

# Load the JSON data
data = json.loads(details)
jatsxml_url = data['collection'][0]['jatsxml']

In [98]:
jatsxml = requests.get(jatsxml_url).content.decode('utf-8')
with open("input_jats.xml", "w") as f:
    f.write(jatsxml)

In [84]:
get_pubtator_bioc_json('33564768')

b'[]'

In [95]:
id = 34210893
url = "https://www-ncbi-nlm-nih-gov.ezproxy.lib.ucalgary.ca/research/bionlp/RESTful/pmcoa.cgi/BioC_json/" + str(id) + "/unicode"
bioc = requests.get(url, allow_redirects=True)
bioc = bioc.content.decode('utf-8')

In [96]:
with open("output_bioc.xml", "w") as f:
    f.write(bioc)

In [ ]:
# Obtain BioC JSON file from PMID or PMC with a maximum of 3 API calls per second

@sleep_and_retry
@limits(calls=3, period=1)
def get_pubtator_bioc_json(id):
    # API link for BioC
    url = "https://www-ncbi-nlm-nih-gov.ezproxy.lib.ucalgary.ca/research/bionlp/RESTful/pmcoa.cgi/BioC_json/" + str(id) + "/unicode"
    bioc = requests.get(url, allow_redirects=True)

    # Failed to grab BioC
    if bioc.status_code != 200:
        raise ConnectionError('could not download {}\nerror code: {}'.format(url, bioc.status_code))
        return None
    
    return bioc.content